google collab depedencies

In [1]:
# !pip -q install bertopic
# !pip -q install sastrawi
# !pip -q install gensim

In [2]:
# !git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4

In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce GTX 1650 Ti


In [5]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")
df = df[df['year'] == 2025]
# df = df[df['bank'] == "LIVIN_MANDIRI_REVIEWS"]
df.head()

,reviewId,bank,score,year,text
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
10,33535e95-15cb-49b3-bcb7-894957cb159d,WONDR_BNI_REVIEWS,1,2025,ngelag mulu deh
12,8091018d-801d-4a9f-a3ff-86491d781b1e,BRIMO_REVIEWS,1,2025,transaksi berhasil uang enggak masuk gimnaa si...
13,658c217f-74b2-4280-b07a-7ab529fd97a1,BCAMOBILE_REVIEWS,2,2025,sering keluar harus verifikasi lagi terus luma...
17,47ed6779-21fd-45bb-a5a6-c6d92ef93186,BCAMOBILE_REVIEWS,1,2025,malu ih bca mah


In [6]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 42,661


In [7]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 42,661


# SIMCSE IndoBERT

In [8]:
from sentence_transformers import SentenceTransformer

# Gunakan SimCSE untuk menekan anisotropy dan merapatkan klaster
embedding_model = SentenceTransformer("LazarusNLP/simcse-indobert-base", device=device)

embeddings = embedding_model.encode(
    documents,
    batch_size=128,             # GPU T4/V100 Colab sanggup menangani batch 128 untuk 60k data
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # Wajib: Memaksa vektor berukuran L2=1 agar Cosine Distance presisi
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/334 [00:00<?, ?it/s]

# BERTopic

In [9]:
# embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (42661, 768)


stop words

In [10]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.corpus import stopwords as nltk_stopwords
from bertopic.vectorizers import ClassTfidfTransformer

sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# Pure stopwords gabungan (NLTK + Sastrawi)
pure_stopwords = list(set(nltk_stopwords.words('indonesian')).union(set(sastrawi_stopwords)))

topic_stopwords = list(set(
    pure_stopwords + [
        "brimo",
        "livin",
        "mandiri",
        "bca",
        "bni",
        "wondr"
    ]
))


vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=topic_stopwords,
    token_pattern=r"(?u)\b[^\d\W]+\b",
    min_df=5  # Untuk 60k data, min_df=5 efektif membuang kata typo langka
)

# Strict c-TF-IDF Transformer untuk memotong frequent words antar-klaster
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True
)

baseline UMAP for testing purpose

In [12]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [13]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

K-Means

In [14]:
# # searching best K value
# reduced_embeddings = umap_model.fit_transform(embeddings)  # pakai UMAP embedding yang sama

# k_range = range(20, 100, 5)
# inertias = []
# silhouettes = []

# for k in k_range:
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     labels = km.fit_predict(reduced_embeddings)
#     inertias.append(km.inertia_)
#     sil = silhouette_score(reduced_embeddings, labels)
#     silhouettes.append(sil)
#     print(f"k={k} -> inertia={km.inertia_:.1f}, silhouette={sil:.4f}")

# fig, ax1 = plt.subplots(figsize=(10,5))
# ax1.plot(k_range, inertias, 'b-o', label='Inertia (Elbow)')
# ax1.set_xlabel('Jumlah Klaster (K)')
# ax1.set_ylabel('Inertia', color='b')

# ax2 = ax1.twinx()
# ax2.plot(k_range, silhouettes, 'r-s', label='Silhouette')
# ax2.set_ylabel('Silhouette Score', color='r')

# plt.title('Elbow Method & Silhouette Score vs K')
# plt.show()

In [15]:
from sklearn.cluster import KMeans

kmeans_model = KMeans(
    n_clusters=70,
    random_state=42,
    n_init=10
)

In [16]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=kmeans_model,
    ctfidf_model=ctfidf_model,
    verbose=True
)

In [17]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-12 14:51:46,159 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-08-12 14:51:50,119 - BERTopic - Dimensionality - Completed ✓
2026-08-12 14:51:50,129 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-12 14:51:53,679 - BERTopic - Cluster - Completed ✓
2026-08-12 14:51:53,690 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-12 14:51:54,768 - BERTopic - Representation - Completed ✓


outliers removal

# Evaluation for Topic Quality

Basic Statistics

In [18]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,0,1280,0_potongan_biaya_ribu_admin,"[potongan, biaya, ribu, admin, biaya admin, po...",[user gpn blue passport admin bulanan tertulis...
1,1,1025,1_mobile_mobile banking_wonder_banking,"[mobile, mobile banking, wonder, banking, apli...",[aplikasi di mobile banking bni yang terbaru t...
2,2,1018,2_uninstall_instal_hapus_download,"[uninstall, instal, hapus, download, aksesibil...",[update terbaru membuat aplikasi tidak dapat d...
3,3,1009,3_login login_login susah_susah login_login,"[login login, login susah, susah login, login,...",[knapa saya mau login enggak bisa layanan tida...
4,4,993,4_kartu debit_debit_atm_kartu atm,"[kartu debit, debit, atm, kartu atm, kartu, te...",[loginnya susah kenapa harus pakai debit card ...
5,5,958,5_pinjaman_asuransi_angsuran_nabung,"[pinjaman, asuransi, angsuran, nabung, potonga...",[jangan pakai bri mau bro saldo gue ada satu j...
6,6,946,6_menanggapi_terdeteksi aman_aplikasi buka_putih,"[menanggapi, terdeteksi aman, aplikasi buka, p...",[ini kenapa ya kok apk brimo tidak bisa dibuka...
7,7,919,7_aplikasinya buka_aplikasi buka_dibuka aplika...,"[aplikasinya buka, aplikasi buka, dibuka aplik...","[kok enggak bisa dibuka ya aplikasi nya, susah..."
8,8,908,8_tim_tim perbaiki_loading_kendala tim,"[tim, tim perbaiki, loading, kendala tim, perb...",[semenjak update 26 des banyak yang mengalami ...
9,9,868,9_pengiriman_masuk rekening_berhasil_proses,"[pengiriman, masuk rekening, berhasil, proses,...",[kirim uang ke luar negeri lewat jalur pengiri...


In [19]:
num_topics = len(
    topic_info[topic_info["Topic"] != -1]
)

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 70
Outliers            : 0
Outlier Percentage  : 0.00%


Topic Size

In [20]:
topic_info[["Topic","Count"]]

,Topic,Count
0,0,1280
1,1,1025
2,2,1018
3,3,1009
4,4,993
...,...,...
65,65,252
66,66,228
67,67,112
68,68,99


Top Words

In [21]:
top_10_topics = topic_model.get_topic_info()
top_10_topics = top_10_topics[top_10_topics.Topic != -1].nlargest(10, "Count")

for _, row in top_10_topics.iterrows():
    topic_id = row['Topic']
    doc_count = row['Count']

    print("=" * 80)
    print(f"TOPIC {topic_id} | JUMLAH DOKUMEN: {doc_count}")
    print("=" * 80)

    # Menampilkan word-score pair bawaan BERTopic (c-TF-IDF scores)
    words_with_scores = topic_model.get_topic(topic_id)
    for word, score in words_with_scores:
        print(f"  - {word:<20} : {score:.4f}")
    print()

TOPIC 0 | JUMLAH DOKUMEN: 1280
  - potongan             : 0.3832
  - biaya                : 0.3588
  - ribu                 : 0.3255
  - admin                : 0.3117
  - biaya admin          : 0.3018
  - potong               : 0.2700
  - bulanan              : 0.2421
  - kena                 : 0.2421
  - perbulan             : 0.2418
  - sisa                 : 0.2388

TOPIC 1 | JUMLAH DOKUMEN: 1025
  - mobile               : 0.3352
  - mobile banking       : 0.3299
  - wonder               : 0.3178
  - banking              : 0.3103
  - aplikasi mobile      : 0.2544
  - wonder by            : 0.2487
  - pakai mobile         : 0.2272
  - aplikasi wonder      : 0.2155
  - banking wonder       : 0.2054
  - mending mobile       : 0.2001

TOPIC 2 | JUMLAH DOKUMEN: 1018
  - uninstall            : 0.2577
  - instal               : 0.2426
  - hapus                : 0.2408
  - download             : 0.2356
  - aksesibilitas        : 0.2350
  - install              : 0.2346
  - uninstal         

Representative Reviews

In [22]:
# Ambil info topik dan urutkan berdasarkan jumlah dokumen terbesar (kecuali outlier -1)
topic_info = topic_model.get_topic_info()
top_10_topics = topic_info[topic_info.Topic != -1].nlargest(10, "Count")["Topic"].tolist()

print("=== TOP 10 TOPIK PALING REPRESENTATIF ===")

for topic_id in top_10_topics:
    # Ambil ukuran klaster asli
    cluster_size = topic_info.loc[topic_info.Topic == topic_id, "Count"].values[0]

    # Ambil kata kunci utama topik untuk mempermudah pembacaan aspek
    keywords = ", ".join([w for w, _ in topic_model.get_topic(topic_id)[:5]])

    # Ambil dokumen yang secara matematis paling dekat dengan centroid klaster (Bawaan BERTopic)
    rep_docs = topic_model.get_representative_docs(topic_id)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id} | CLUSTER SIZE: {cluster_size}")
    print(f"KEYWORDS : {keywords}")
    print("=" * 120)

    # BERTopic menyimpan maksimum 3 representative docs per topik secara default
    for i, doc in enumerate(rep_docs, 1):
        print(f"{i}. {doc}")

=== TOP 10 TOPIK PALING REPRESENTATIF ===

TOPIC 0 | CLUSTER SIZE: 1280
KEYWORDS : potongan, biaya, ribu, admin, biaya admin
1. user gpn blue passport admin bulanan tertulis 14 15 ribu saja tapi sudah 2 minggu ini setiap ada dana masuk potongan admin sekitar 42rb dan saya tidak dapat meminta detail rincian biaya admin nya ada apa
2. lintah darat memang mandiri setiap bulan kena potongan terus biaya administarasi rekening lah biaya administrasi kartu debit lah biaya saldo minimum lah gila lintah darat
3. begini ya bri kenapa sekarang biaya admin sangat mahal tiba-tiba saja belum ada sebulan sudah kena potongan 39 0 belum biaya admin lain2 dari bri mau urus secara offline ke bank juga kena admin 10 0 potong dalam 50 0 saldo isi cuma 60ribu sudah habis kepotong tiap hari lama lama mahal

TOPIC 1 | CLUSTER SIZE: 1025
KEYWORDS : mobile, mobile banking, wonder, banking, aplikasi mobile
1. aplikasi di mobile banking bni yang terbaru ternyata kalah saing dengan mobile banking bank sebelah yang

silhoutte score

In [23]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.3168


In [24]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]

    topic_words.append(words)

flat_words = list(chain.from_iterable(topic_words))

unique_words = len(set(flat_words))
total_words = len(flat_words)

topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.7957


NPMI

In [25]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [26]:
doc.split()

['keterangan',
 'transfer',
 'berhasil',
 'namun',
 'dana',
 'nya',
 'tidak',
 'masuk',
 'ke',
 'rekening',
 'tujuan',
 'dan',
 'saldo',
 'sudah',
 'terpotong',
 'mana',
 'duit',
 'nya',
 'perlu',
 'untuk',
 'di',
 'gunakan',
 'nelpon',
 'customer',
 'service',
 'habis',
 'pulsa',
 '30',
 'ribu',
 'lebih',
 'dan',
 'menunggu',
 'sampai',
 '5',
 'hari',
 'kerja',
 'kata',
 'nya',
 'lambat',
 'banget']

In [27]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [28]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:

    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
        if word in dictionary.token2id
    ]

    if len(words) >= 2:
        topic_words.append(words)

print(f"Valid Topics for NPMI: {len(topic_words)}")

Valid Topics for NPMI: 70


In [29]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.0346


In [30]:
per_topic_npmi = coherence_model.get_coherence_per_topic()

npmi_df = pd.DataFrame({
    "Topic": [
        t for t in topic_info["Topic"]
        if t != -1
    ],
    "NPMI": per_topic_npmi
})

npmi_df.sort_values("NPMI", ascending=False).head(10)

,Topic,NPMI
53,53,0.410559
40,40,0.398413
61,61,0.315168
14,14,0.278706
16,16,0.266858
31,31,0.253510
66,66,0.245135
0,0,0.236880
56,56,0.220105
52,52,0.217156


In [31]:
worst_topics = (
    npmi_df
    .sort_values("NPMI", ascending=True)
    .head(10)["Topic"]
    .tolist()
)

for topic_id in worst_topics:
    print("\n" + "=" * 80)
    print(f"TOPIC {topic_id}")
    print("=" * 80)

    print("NPMI:",
          npmi_df.loc[
              npmi_df["Topic"] == topic_id,
              "NPMI"
          ].iloc[0]
    )

    print("Words:")
    print(topic_model.get_topic(topic_id))

    print("\nRepresentative documents:")
    print(
        topic_model.get_representative_docs(topic_id)[:5]
    )


TOPIC 36
NPMI: -0.33092763810974674
Words:
[('buka masuk', np.float64(0.44207524354268457)), ('masuk susah', np.float64(0.39150463004970143)), ('living', np.float64(0.3757058652590756)), ('masuk buka', np.float64(0.3548056941237911)), ('buka buka', np.float64(0.35427498332535556)), ('dibuka', np.float64(0.3397228973881523)), ('perbaharui', np.float64(0.3317755188151919)), ('masuk dibuka', np.float64(0.32470574815603614)), ('dibuka dibuka', np.float64(0.2968517414600521)), ('buka mbanking', np.float64(0.29606067089461563))]

Representative documents:
['aplikasi busuk mau masuk saja susah benar enakan yang dulu', 'kenapa indak bisa di buka living mandirinya ada tulisan akun anda belum dapat di akses', 'living sekarang susah sekali mau buka sedikit sedikit keluar sendiri sudah enggak bagus lagi enggak seperti dulu']

TOPIC 60
NPMI: -0.29129712875967445
Words:
[('sampah', np.float64(0.3892024454847253)), ('aplikasi sampah', np.float64(0.3291044095307531)), ('apk sampah', np.float64(0.3217

In [32]:
per_topic = np.array(
    coherence_model.get_coherence_per_topic()
)

overall = coherence_model.get_coherence()

print("Gensim overall :", overall)
print("Mean per-topic :", per_topic.mean())
print("Difference     :", overall - per_topic.mean())

Gensim overall : 0.03458062337711526
Mean per-topic : 0.03458062337711526
Difference     : 0.0


# Evaluations Summary

# Evaluation for Clustering Quality


DBCV -> Only if using HDBSCAN method

In [33]:
# mask = np.array(topics) != -1
# X = topic_model.umap_model.embedding_[mask].astype(np.float64)
# labels = np.array(topics)[mask]

# dbcv_score = validity_index(X, labels)
# print(f"DBCV : {dbcv_score:.4f}")

In [34]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS            29.27
WONDR_BNI_REVIEWS        26.60
LIVIN_MANDIRI_REVIEWS    26.05
BCAMOBILE_REVIEWS        18.09
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  WONDR_BNI_REVIEWS  LIVIN_MANDIRI_REVIEWS  \
topic                                                            
35          3.247921           0.051136               0.078319   
49          3.279355           0.060536               0.030713   
30          0.294862           0.584305               2.320402   
31          0.746892           1.513401               0.706410   
33          0.465697           1.358489               0.942903   
...              ...                ...                    ...   
44          0.873310           0.932857               0.995884   
45          0.860668           1.310361               1.155978   
15          1.333199           0.871193               0.945591   
29          1.099838           0.912177               0.919484 

checking outliers

In [35]:
# import itertools

# param_grid = {
#     "min_cluster_size": [30, 50, 75],
#     "min_samples": [10, 15, 20],
#     "cluster_selection_method": ["eom", "leaf"],
# }

# results = []
# combos = list(itertools.product(*param_grid.values()))
# print(f"Total kombinasi: {len(combos)}")

# for mcs, ms, method in combos:
#     hdbscan_test = HDBSCAN(
#         min_cluster_size=mcs, min_samples=ms, metric="euclidean",
#         cluster_selection_method=method, prediction_data=True,
#     )
#     tm = BERTopic(
#         embedding_model=None, calculate_probabilities=False,
#         vectorizer_model=vectorizer_model, umap_model=umap_model,
#         hdbscan_model=hdbscan_test, verbose=False,
#     )
#     tpcs, _ = tm.fit_transform(documents, embeddings)

#     ti = tm.get_topic_info()
#     n_topics = len(ti) - 1
#     outlier_pct = (np.array(tpcs) == -1).sum() / len(tpcs) * 100
#     max_share = ti[ti.Topic != -1]["Count"].max() / len(tpcs) * 100 if n_topics > 0 else 0
#     mask = np.array(tpcs) != -1
#     sil = silhouette_score(tm.umap_model.embedding_[mask], np.array(tpcs)[mask]) if len(set(np.array(tpcs)[mask])) > 1 else float("nan")

#     row = {"min_cluster_size": mcs, "min_samples": ms, "method": method,
#            "topics": n_topics, "outlier_%": round(outlier_pct, 2),
#            "max_topic_share_%": round(max_share, 2), "silhouette": round(sil, 4)}
#     results.append(row)
#     print(row)

# results_df = pd.DataFrame(results).sort_values("outlier_%")
# results_df